In [5]:
import json
import pandas as pd
from utils import compare_preds

In [6]:
ENTITIES_TO_PREDICT = ["HouseNumber", "StreetName","Neighborhood", "City", "District", "State", "Country"]

csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)

# Load cached predictions
with open("cached_preds_val.json", "r") as f:
    cached_preds = json.load(f)

In [7]:
CONFIG_NAME = "Llama-3-8B-prompt2-similar3shot"

In [8]:
# Compute metrics for the selected config only
if CONFIG_NAME not in cached_preds:
    raise KeyError(f"Config '{CONFIG_NAME}' not found in cached_preds. Available keys: {list(cached_preds.keys())[:5]} ...")

preds = cached_preds[CONFIG_NAME]["preds"]
preds_df = pd.DataFrame(preds)
metrics = compare_preds(preds_df, bzkopen_val, target_columns=ENTITIES_TO_PREDICT)
print(f"Metrics for {CONFIG_NAME}:")
print(metrics)

Metrics for Llama-3-8B-prompt2-similar3shot:
OrderedDict([('accuracy', np.float64(0.6578947368421053)), ('precision', np.float64(0.8413173652694611)), ('recall', np.float64(0.7698630136986301)), ('f1', np.float64(0.804005722460658)), ('accuracy_with_tol_1', np.float64(0.6616541353383458)), ('accuracy_with_tol_2', np.float64(0.668233082706767)), ('accuracy_with_tol_3', np.float64(0.6729323308270677)), ('accuracy_with_tol_4', np.float64(0.6804511278195489)), ('average_levenshtein', np.float64(0.7265037593984962)), ('average_similarity', np.float64(0.6705231729320105)), ('average_levenshtein_match', np.float64(0.5034387895460798)), ('average_similarity_match', np.float64(0.9813434057766975)), ('no_match_rate', np.float64(0.31672932330827064))])


In [9]:
# Helper to show all error cases for a specific config/entity

def show_errors_for_config_entity(config_name, entity):
    """Display prediction errors for a given config and entity."""
    if config_name not in cached_preds:
        print(f"Config '{config_name}' not found in cached_preds.")
        return

    preds = cached_preds[config_name]["preds"]
    preds_df = pd.DataFrame(preds)

    if entity not in preds_df.columns:
        print(f"Entity '{entity}' not found in predictions.")
        return

    mismatch_mask = preds_df[entity] != bzkopen_val[entity].values
    error_cases = pd.DataFrame({
        "Address": bzkopen_val.loc[mismatch_mask, "FullAddress"].values,
        "Predicted": preds_df.loc[mismatch_mask, entity].values,
        "Actual": bzkopen_val.loc[mismatch_mask, entity].values,
    })

    print(f"\nErrors for {config_name} / {entity}:")
    print(error_cases.to_string(index=False))
    print(f"\nTotal errors: {len(error_cases)}")
    return error_cases

In [10]:
# Wrapper: use CONFIG_NAME by default

def show_errors_for_selected_config(entity):
    return show_errors_for_config_entity(CONFIG_NAME, entity)

In [11]:
df = show_errors_for_selected_config("State")


Errors for Llama-3-8B-prompt2-similar3shot / State:
                                              Address     Predicted Actual
                            Regensburg, Königstr. 2/I           NaN    NaN
                                             Dortmund           NaN    NaN
                        Nürnberg, Nibelungenstrasse 8           NaN    NaN
                         Jöhlingen/Krs.Durlach/Baden.   Krs.Durlach    NaN
            8 Burlington Road, Manchester 20/England.           NaN    NaN
                                    Leer/Ostfriesland           NaN    NaN
                                             Hannover           NaN    NaN
                    New York, USA, 220 West 98th Str.           NaN    NaN
                                               Berlin           NaN    NaN
Berlin-Lichterfelde-Ost, Berlinerstr.48a (Altersheim)           NaN    NaN
                                              Krefeld           NaN    NaN
                            Nürschan, Bez.Mies/

In [12]:
df

,Address,Predicted,Actual
0,"Regensburg, Königstr. 2/I",NaN,NaN
1,Dortmund,NaN,NaN
2,"Nürnberg, Nibelungenstrasse 8",NaN,NaN
3,Jöhlingen/Krs.Durlach/Baden.,Krs.Durlach,NaN
4,"8 Burlington Road, Manchester 20/England.",NaN,NaN
...,...,...,...
140,Sosnowice/Polen,NaN,NaN
141,2114-79 St. Jackson Heights. N.Y. USA,N.Y.,NaN
142,Losone CSR,NaN,NaN
143,Rum.,NaN,NaN


In [13]:
# Single-address test with Llama-3-8B-prompt2-similar3shot

from mllms import SimilarExamples, JsonDictPromptTemplate
import pandas as pd
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#  Load training data used for SimilarExamples
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])
bzkopen_train = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)

#  Prompt 2 definition
llama_prompt2 = JsonDictPromptTemplate(
    "You are a german archivist handling the digitalization of german documents from the compensation efforts that followed the second world war. "
    "Your current task consists of annotating addresses found in the archival documents, identifying the respective components of each address. "
    "Consider the component types: " + ", ".join(ENTITIES_TO_PREDICT + ["Other"]) + ". "
    "It is essential that you remain loyal to the original text and do not add any information not explictly mentioned in the address. "
    "Addresses will most times be written in german, meaning country and city names may be in german. The addresses may include german terms such as:\n"
    " - \"burg\" or \"stadt\" for city\n"
    " - \"straße\", \"avenue\" or its abbreviation \"str.\" and \"av.\" for street.\n"
    "These terms may occur as a suffix to another word.\n"
    "Format the output as a JSON object with the component types as keys.\n%(examples)s"
    "Now annotate the following address:\n%(address)s"
)

# 4) SimilarExamples config for "similar3shot" (num_examples=3)
similar3shot_cfg = {
    "factory": SimilarExamples,
    "factory_kargs": dict(
        example_addresses=bzkopen_train["FullAddress"],
        example_labels=bzkopen_train,
        num_examples=3,
        labels_to_include=ENTITIES_TO_PREDICT,
        device=device,
    ),
}


In [ ]:

from augmentation import GeoNamesLookup, augment_dataset

GEONAMES_USER = "amelgd"  

geo = GeoNamesLookup(username=GEONAMES_USER)


augmented = augment_dataset(
    df         = bzkopen_train,
    geo        = geo,
    n_augments = 3,
    seed       = 42,
)

has_label = bzkopen_train["City"].notna() | bzkopen_train["State"].notna()
print(f"Original rows with City/State labels : {has_label.sum()}")
print(f"New synthetic rows generated         : {len(augmented)}")

augmented[["FullAddress", "City", "State", "Country", "is_augmented", "source_address"]]


Fetching city pool for 'USA' (US) …
  → 1000 cities
Fetching city pool for 'Polen' (PL) …
  → 1000 cities
Fetching city pool for 'Israel' (IL) …
  → 1000 cities
Fetching city pool for 'Griechenland' (GR) …
  → 1000 cities
Fetching city pool for 'Frankreich' (FR) …
  → 1000 cities
Fetching city pool for 'Schweiz' (CH) …
  → 1000 cities
Fetching city pool for 'Italien' (IT) …
  → 1000 cities
Fetching city pool for 'Rumänien' (RO) …
  → 1000 cities
Fetching city pool for 'Jug.' (RS) …
  → 1000 cities
Fetching city pool for 'Bulg.' (BG) …
  → 1000 cities
Fetching city pool for 'Ungarn' (HU) …
  → 1000 cities
Fetching city pool for 'Russland' (RU) …
  → 1000 cities
Fetching city pool for 'CSR' (CZ) …
  → 1000 cities
Fetching city pool for 'Ukraine' (UA) …
  → 1000 cities
Fetching city pool for 'LATVIA' (LV) …
  → 1000 cities
Fetching city pool for 'Litauen' (LT) …
  → 1000 cities
Original rows with City/State labels : 726
New synthetic rows generated         : 1290


,FullAddress,City,State,Country,is_augmented,source_address
0,Ruskowa,Ruskowa,NaN,NaN,False,NaN
1,"170 Clymer Street, Brooklyn, N.Y. USA",N.Y.,NaN,USA,False,NaN
2,Drohobycz,Drohobycz,NaN,NaN,False,NaN
3,"London N.W.3/England, 109 Green Hill",London,NaN,NaN,False,NaN
4,Dresden,Dresden,NaN,NaN,False,NaN
...,...,...,...,...,...,...
1285,Žebrák / CSR,Žebrák,NaN,CSR,True,Pilzen / CSR
1286,Mikulovice / CSR,Mikulovice,NaN,CSR,True,Pilzen / CSR
1287,"311 Lincoln Rd., San Antonio, Florida/USA",San Antonio,Florida,USA,True,"311 Lincoln Rd., Miami Beach, Florida/USA"
1288,"311 Lincoln Rd., Youngstown, Florida/USA",Youngstown,Florida,USA,True,"311 Lincoln Rd., Miami Beach, Florida/USA"


In [15]:
from mllms import SimilarExamples
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])
bzkopen_train = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)

few_shot_strategy = SimilarExamples(
    example_addresses=bzkopen_train["FullAddress"],
    example_labels=bzkopen_train,
    num_examples=3,
    labels_to_include=ENTITIES_TO_PREDICT,
    device=device,
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:

def visualize_few_shots(address, example_strategy=None, prompt=None):
    if example_strategy is None:
        example_strategy = few_shot_strategy

    examples, metadatas = example_strategy.find_examples(address)
    has_breakdown = metadatas and "pattern_score" in metadatas[0]
    sec_key = next((k for k in ("token_score", "embedding_score") if metadatas and k in metadatas[0]), None)
    sec_label = {"token_score": "token", "embedding_score": "emb"}.get(sec_key, "")

    print(f"Query:   {address!r}")
    if metadatas and "query_pattern" in metadatas[0]:
        print(f"Pattern: {metadatas[0]['query_pattern']!r}")
    print()

    for m in metadatas:
        score_str = f"score={m['score']:.4f}"
        if has_breakdown:
            score_str += f"  (pattern={m['pattern_score']:.4f}, {sec_label}={m[sec_key]:.4f})"
        print(f"{score_str}")
        print(f"       address: {m['address']!r}")
        if "example_pattern" in m:
            print(f"       pattern: {m['example_pattern']!r}")
        labels = {k: v for k, v in m["example"].items() if v}
        print(f"       labels:  {labels}")
        print()

    if prompt is not None:
        print(prompt.make_prompt(address, examples))


In [17]:

# pattern hybrid strategy
from mllms import address_to_pattern

for addr in ["Karl-Marx-Straße 21/3, Berlin", "Krs. Breslau, Hauptstr. 4", "Haspe-Hagen"]:
    print(f"{addr!r:45s} → {address_to_pattern(addr)!r}")


'Karl-Marx-Straße 21/3, Berlin'               → 'WORD-WORD-STREET NUM/NUM, WORD'
'Krs. Breslau, Hauptstr. 4'                   → 'DIST WORD, WORD NUM'
'Haspe-Hagen'                                 → 'WORD-WORD'


In [61]:
# Load hybrid3shot predictions
with open("preds_hybrid_ner3shot_val.json") as f:
    hybrid_data = json.load(f)

hybrid_preds = hybrid_data["preds"]
hybrid_df = pd.DataFrame([
    {k: v for k, v in p.items() if k not in ("fullConversation", "___example_metadata")}
    for p in hybrid_preds
])
hybrid_df["FullAddress"] = bzkopen_val["FullAddress"].values

print(f"Loaded {len(hybrid_df)} predictions")
print(f"Columns: {list(hybrid_df.columns)}")
hybrid_df.head()

Loaded 152 predictions
Columns: ['City', 'StreetName', 'HouseNumber', 'District', 'State', 'Country', 'Other', 'error', 'Town', 'FullAddress']


,City,StreetName,HouseNumber,District,State,Country,Other,error,Town,FullAddress
0,Regensburg,Königstr.,2,NaN,NaN,NaN,NaN,NaN,NaN,"Regensburg, Königstr. 2/I"
1,Dortmund,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dortmund
2,Nürnberg,Nibelungenstrasse,8,NaN,NaN,NaN,NaN,NaN,NaN,"Nürnberg, Nibelungenstrasse 8"
3,Jöhlingen,NaN,NaN,Durlach,Baden,NaN,NaN,NaN,NaN,Jöhlingen/Krs.Durlach/Baden.
4,Manchester,Burlington Road,8,NaN,NaN,England,NaN,NaN,NaN,"8 Burlington Road, Manchester 20/England."


In [ ]:
def show_pred(i):
    """Show full details for prediction at index i"""
    pred = hybrid_preds[i]
    gt = bzkopen_val.iloc[i]

    full_address = gt["FullAddress"]
    print(f"[{i}/{len(hybrid_preds)-1}] Address: {full_address!r}")

    metadata = pred.get("___example_metadata", [])
    stored_pattern = next((m["query_pattern"] for m in metadata if m.get("query_pattern")), None)
    if stored_pattern:
        print(f"Pattern: {stored_pattern!r}")

    print()
    print("Ground truth:")
    gt_labels = {col: gt[col] for col in ENTITIES_TO_PREDICT if pd.notna(gt[col]) and gt[col] != ""}
    print(f"  {gt_labels}")

    print()
    print("Prediction:")
    pred_labels = {col: pred[col] for col in ENTITIES_TO_PREDICT if col in pred and pd.notna(pred[col]) and pred[col] != ""}
    print(f"  {pred_labels}")

    correct = {k for k in pred_labels if gt_labels.get(k) == pred_labels[k]}
    wrong   = {k for k in pred_labels if k not in correct}
    missed  = {k for k in gt_labels if k not in pred_labels}
    if correct: print(f"  ✓ correct : {correct}")
    if wrong:   print(f"  ✗ wrong   : { {k: (pred_labels[k], gt_labels.get(k)) for k in wrong} }")
    if missed:  print(f"  - missed  : {missed}")

    print()
    print("Few-shot examples:")
    for j, m in enumerate(metadata):
        score_str = f"score={m['score']:.4f}"
        if "pattern_score" in m:
            tok_key = next((k for k in ("token_score", "embedding_score") if k in m), None)
            if tok_key:
                tok_label = "token" if tok_key == "token_score" else "emb"
                score_str += f"  (pattern={m['pattern_score']:.4f}, {tok_label}={m[tok_key]:.4f})"
        included = m.get("included", True)
        print(f"  [{j+1}] {score_str}{'' if included else '  [excluded]'}")
        print(f"       address: {m['address']!r}")
        if "example_pattern" in m:
            print(f"       pattern: {m['example_pattern']!r}")
        ex_labels = {k: v for k, v in m["example"].items() if v}
        print(f"       labels:  {ex_labels}")




In [ ]:
show_pred(14)

[14/151] Address: 'Haspe-Hagen'
Pattern: 'City-City'

Ground truth:
  {'Neighborhood': 'Haspe', 'City': 'Hagen'}

Prediction:
  {'City': 'Haspe'}
  ✗ wrong   : {'City': ('Haspe', 'Hagen')}
  - missed  : {'Neighborhood'}

Few-shot examples:
  [1] score=1.0000
       address: 'Kerprich-Hemmersdorf'
       pattern: 'City-City'
       labels:  {'City': 'Kerprich-Hemmersdorf'}
  [2] score=1.0000
       address: 'Dortmund-Lütgendortmund'
       pattern: 'City-City'
       labels:  {'City': 'Dortmund'}
  [3] score=1.0000
       address: 'Hbg-Bergedorf'
       pattern: 'City-City'
       labels:  {'City': 'Bergedorf'}


In [19]:
import re
import pandas as pd
from pathlib import Path

# ── Parse a metrics txt file ────────────────────────────────────────────────
def parse_metrics_txt(path):
    text = Path(path).read_text()
    kv = {}
    for line in text.splitlines():
        m = re.match(r'\s+(\S.*?)\s{2,}([\d.]+)\s*$', line)
        if m:
            kv[m.group(1).strip()] = float(m.group(2))
    return kv

FILES = {
    "Embedding 3-shot":                  "metrics_similar3shot_val.txt",
    "Pattern 3-shot":                    "metrics_patternsim3shot_val.txt",
    "Hybrid 3-shot (2-emb + 1-pat*)":    "metrics_hybrid3shot_val.txt",
    "Hybrid 3-shot (1-emb + 2-pat)":     "metrics_hybrid2p1e_val.txt",
    "Embedding 3-shot -District":        "metrics_similar3shot_v2_val.txt",
    "Pattern 3-shot -District":          "metrics_patternsim3shot_v2_val.txt",
    "Hybrid 3-shot (2-emb + 1-pat*) -District": "metrics_hybrid3shot_v2_val.txt",
    "Hybrid 3-shot (1-emb + 2-pat) -District":  "metrics_hybrid2p1e_v2_val.txt",
}
# * pattern shot only included when pattern score = 1.0 (exact structural match), otherwise falls back to 3rd embedding shot

OVERALL_COLS   = ["accuracy", "precision", "recall", "f1", "average_similarity"]
ENTITY_COLS    = ["HouseNumber", "StreetName", "City", "District", "State", "Country"]

rows = {}
for name, fname in FILES.items():
    p = Path(fname)
    if p.exists():
        rows[name] = parse_metrics_txt(p)

df = pd.DataFrame(rows).T
df.index.name = "config"

# ── Overall metrics table ────────────────────────────────────────────────────
overall = df[[c for c in OVERALL_COLS if c in df.columns]].copy()
overall = overall.astype(float).round(4)

print("Overall metrics (val set, District excluded from aggregation where applicable)")
display(
    overall.style
    .format("{:.4f}")
    .highlight_max(axis=0, props="font-weight:bold;color:green")
    .highlight_min(axis=0, props="color:lightgrey")
)

# ── Per-entity F1 table ──────────────────────────────────────────────────────
entity_f1 = df[[c for c in ENTITY_COLS if c in df.columns]].copy()
entity_f1 = entity_f1.astype(float).round(4)

print("\nPer-entity F1")
display(
    entity_f1.style
    .format("{:.4f}", na_rep="-")
    .highlight_max(axis=0, props="font-weight:bold;color:green")
    .highlight_min(axis=0, props="color:lightgrey")
)


Overall metrics (val set, District excluded from aggregation where applicable)


,accuracy,precision,recall,f1,average_similarity
config,,,,,
Embedding 3-shot,0.9039,0.8041,0.8648,0.8333,0.9231
Pattern 3-shot,0.8987,0.7988,0.8491,0.8232,0.9196
Hybrid 3-shot (2-emb + 1-pat*),0.9066,0.8149,0.8585,0.8361,0.9265
Hybrid 3-shot (1-emb + 2-pat),0.9079,0.8201,0.8742,0.8463,0.9250
Embedding 3-shot -District,0.9211,0.8514,0.8648,0.8580,0.9389
Pattern 3-shot -District,0.9132,0.8288,0.8679,0.8479,0.9324
Hybrid 3-shot (2-emb + 1-pat*) -District,0.9250,0.8450,0.8742,0.8594,0.9446
Hybrid 3-shot (1-emb + 2-pat) -District,0.9276,0.8669,0.8805,0.8736,0.9428



Per-entity F1


,HouseNumber,StreetName,City,District,State,Country
config,,,,,,
Embedding 3-shot,0.8226,0.8548,0.8464,-,0.5000,0.8791
Pattern 3-shot,0.8000,0.9091,0.8367,-,0.3333,0.8571
Hybrid 3-shot (2-emb + 1-pat*),0.8525,0.8689,0.8356,-,0.4000,0.9195
Hybrid 3-shot (1-emb + 2-pat),0.8500,0.8926,0.8532,-,0.3871,0.9130
Embedding 3-shot -District,0.8430,0.8739,0.8660,0.2593,0.6087,0.8966
Pattern 3-shot -District,0.8130,0.8871,0.8621,0.5902,0.4444,0.9195
Hybrid 3-shot (2-emb + 1-pat*) -District,0.8525,0.8760,0.8746,0.3729,0.5000,0.8941
Hybrid 3-shot (1-emb + 2-pat) -District,0.8167,0.8926,0.8904,0.5000,0.6087,0.9412
